# Proyecto de Minería de Datos  
## Impacto de COVID-19 en la dinámica urbana de la Ciudad de México

**Objetivo del avance:** cargar, inspeccionar y preparar las fuentes de movilidad, seguridad, turismo, economía y contexto epidemiológico antes de realizar la integración final.

### Periodos de trabajo
- **Pre-COVID:** 2019
- **COVID:** 2020–2021
- **Recuperación:** 2022–2024

> Nota metodológica: para la comparación integrada se trabajará con periodicidad **mensual**, ya que es la mayor granularidad común natural entre las fuentes. Las fuentes con mayor granularidad se conservan también para análisis individuales.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RUTA = Path(".")


## 1. Archivos de entrada

Coloca los cinco CSV en la misma carpeta que este notebook o modifica las rutas siguientes.


In [2]:
ARCHIVOS = {
    "metro": RUTA / "Afluencia.csv",
    "seguridad": RUTA / "carpetasFGJ_acumulado_2025_01.csv",
    "covid": RUTA / "COVID.csv",
    "economia": RUTA / "Economia.csv",
    "turismo": RUTA / "Turismo.csv",
}

for nombre, ruta in ARCHIVOS.items():
    print(f"{nombre:10s} -> {ruta} | existe: {ruta.exists()}")


metro      -> Afluencia.csv | existe: True
seguridad  -> carpetasFGJ_acumulado_2025_01.csv | existe: True
covid      -> COVID.csv | existe: True
economia   -> Economia.csv | existe: True
turismo    -> Turismo.csv | existe: True


## 2. Carga de datos

Para **seguridad** se leen únicamente las columnas necesarias. Esto reduce considerablemente el consumo de memoria sin modificar el archivo original.


In [3]:
metro = pd.read_csv(ARCHIVOS["metro"], low_memory=False)

columnas_seguridad = [
    "anio_inicio",
    "mes_inicio",
    "fecha_inicio",
    "delito",
    "categoria_delito",
    "alcaldia_hecho",
]

seguridad = pd.read_csv(
    ARCHIVOS["seguridad"],
    usecols=columnas_seguridad,
    low_memory=False
)

covid = pd.read_csv(ARCHIVOS["covid"], low_memory=False)
economia = pd.read_csv(ARCHIVOS["economia"], low_memory=False)
turismo = pd.read_csv(ARCHIVOS["turismo"], low_memory=False)

print("Carga finalizada.")


Carga finalizada.


## 3. Inspección inicial

Esta sección se ejecuta **antes de limpiar** los datos para documentar sus características originales: dimensiones, columnas, tipos, valores faltantes y duplicados.


In [4]:
def inspeccionar(nombre, df):
    print("=" * 80)
    print(nombre.upper())
    print("=" * 80)
    print(f"Filas: {df.shape[0]:,}")
    print(f"Columnas: {df.shape[1]}")
    print("\nColumnas:")
    print(df.columns.tolist())
    print("\nTipos:")
    print(df.dtypes)
    print("\nValores faltantes:")
    print(df.isna().sum())
    print(f"\nDuplicados exactos: {df.duplicated().sum():,}")
    print("\nPrimeros registros:")
    display(df.head())

for nombre, df in {
    "Metro": metro,
    "Seguridad": seguridad,
    "COVID": covid,
    "Economía": economia,
    "Turismo": turismo,
}.items():
    inspeccionar(nombre, df)


METRO
Filas: 498,615
Columnas: 6

Columnas:
['fecha', 'anio', 'mes', 'linea', 'estacion', 'afluencia']

Tipos:
fecha        object
anio          int64
mes          object
linea        object
estacion     object
afluencia     int64
dtype: object

Valores faltantes:
fecha        0
anio         0
mes          0
linea        0
estacion     0
afluencia    0
dtype: int64

Duplicados exactos: 0

Primeros registros:


,fecha,anio,mes,linea,estacion,afluencia
0,2018-01-01,2018,Enero,Linea 1,PantitlÃ¡n,16979
1,2018-01-01,2018,Enero,Linea 1,Balbuena,3739
2,2018-01-01,2018,Enero,Linea 1,Isabel la CatÃ³lica,7718
3,2018-01-01,2018,Enero,Linea 1,Candelaria,7775
4,2018-01-01,2018,Enero,Linea 1,Merced,15175


SEGURIDAD
Filas: 758,873
Columnas: 6

Columnas:
['anio_inicio', 'mes_inicio', 'fecha_inicio', 'delito', 'categoria_delito', 'alcaldia_hecho']

Tipos:
anio_inicio          int64
mes_inicio          object
fecha_inicio        object
delito              object
categoria_delito    object
alcaldia_hecho      object
dtype: object

Valores faltantes:
anio_inicio            0
mes_inicio             0
fecha_inicio           2
delito                 0
categoria_delito       1
alcaldia_hecho      7253
dtype: int64

Duplicados exactos: 342,747

Primeros registros:


,anio_inicio,mes_inicio,fecha_inicio,delito,categoria_delito,alcaldia_hecho
0,2019,Enero,2019-01-01,POSESION DE VEHICULO ROBADO,DELITO DE BAJO IMPACTO,LA MAGDALENA CONTRERAS
1,2019,Enero,2019-01-01,ROBO A CASA HABITACION SIN VIOLENCIA,DELITO DE BAJO IMPACTO,BENITO JUAREZ
2,2019,Enero,2019-01-01,NEGACION DEL SERVICIO PUBLICO,DELITO DE BAJO IMPACTO,CUAUHTEMOC
3,2019,Enero,2019-01-01,ROBO DE VEHICULO DE SERVICIO PARTICULAR CON VI...,ROBO DE VEHÍCULO CON Y SIN VIOLENCIA,IZTAPALAPA
4,2019,Enero,2019-01-01,ROBO A PASAJERO / CONDUCTOR DE VEHICULO CON VI...,DELITO DE BAJO IMPACTO,IZTAPALAPA


COVID
Filas: 670
Columnas: 9

Columnas:
['fecha', 'anio', 'mes', 'hospitalizados_totales', 'hospitalizados_totales_cdmx', 'hospitalizados_totales_edomex', 'camas_intubados_totales', 'camas_intubados_cdmx', 'camas_intubados_edomex']

Tipos:
fecha                            object
anio                              int64
mes                              object
hospitalizados_totales            int64
hospitalizados_totales_cdmx       int64
hospitalizados_totales_edomex     int64
camas_intubados_totales           int64
camas_intubados_cdmx              int64
camas_intubados_edomex            int64
dtype: object

Valores faltantes:
fecha                            0
anio                             0
mes                              0
hospitalizados_totales           0
hospitalizados_totales_cdmx      0
hospitalizados_totales_edomex    0
camas_intubados_totales          0
camas_intubados_cdmx             0
camas_intubados_edomex           0
dtype: int64

Duplicados exactos: 1

Primeros regis

,fecha,anio,mes,hospitalizados_totales,hospitalizados_totales_cdmx,hospitalizados_totales_edomex,camas_intubados_totales,camas_intubados_cdmx,camas_intubados_edomex
0,2020-03-24,2020,marzo,50,50,0,39,39,0
1,2020-03-25,2020,marzo,105,105,0,33,33,0
2,2020-03-26,2020,marzo,128,128,0,42,42,0
3,2020-03-27,2020,marzo,175,175,0,60,60,0
4,2020-03-28,2020,marzo,257,257,0,78,78,0


ECONOMÍA
Filas: 68
Columnas: 2

Columnas:
['fecha', 'ciudad_de_mexico']

Tipos:
fecha               object
ciudad_de_mexico     int64
dtype: object

Valores faltantes:
fecha               0
ciudad_de_mexico    0
dtype: int64

Duplicados exactos: 0

Primeros registros:


,fecha,ciudad_de_mexico
0,2019-02-28,14884
1,2019-03-31,12341
2,2019-04-30,6319
3,2019-05-31,21197
4,2019-06-30,2886


TURISMO
Filas: 70
Columnas: 2

Columnas:
['fecha', 'ciudad_de_mexico']

Tipos:
fecha                object
ciudad_de_mexico    float64
dtype: object

Valores faltantes:
fecha               0
ciudad_de_mexico    0
dtype: int64

Duplicados exactos: 0

Primeros registros:


,fecha,ciudad_de_mexico
0,2018-10-31,73.17
1,2018-11-30,75.58
2,2018-12-31,65.45
3,2019-01-31,53.13
4,2019-02-28,69.30


## 4. Conversión y validación de fechas

El archivo de seguridad contiene fechas con más de un formato, por lo que se usa `format="mixed"` para interpretarlas correctamente.


In [13]:
metro["fecha"] = pd.to_datetime(metro["fecha"], errors="coerce")
seguridad["fecha_inicio"] = pd.to_datetime(
    seguridad["fecha_inicio"],
    format="mixed",
    errors="coerce"
)
covid["fecha"] = pd.to_datetime(covid["fecha"], errors="coerce")
economia["fecha"] = pd.to_datetime(economia["fecha"], errors="coerce")
turismo["fecha"] = pd.to_datetime(turismo["fecha"], errors="coerce")

rangos = pd.DataFrame({
    "dataset": ["Metro", "Seguridad", "COVID", "Economía", "Turismo"],
    "fecha_min": [
        metro["fecha"].min(),
        seguridad["fecha_inicio"].min(),
        covid["fecha"].min(),
        economia["fecha"].min(),
        turismo["fecha"].min(),
    ],
    "fecha_max": [
        metro["fecha"].max(),
        seguridad["fecha_inicio"].max(),
        covid["fecha"].max(),
        economia["fecha"].max(),
        turismo["fecha"].max(),
    ],
})

rangos

print("=== COBERTURA TEMPORAL ===")

print(
    "Metro:",
    metro["fecha"].min(),
    "→",
    metro["fecha"].max()
)

print(
    "Seguridad:",
    seguridad["fecha_inicio"].min(),
    "→",
    seguridad["fecha_inicio"].max()
)

print(
    "COVID:",
    covid["fecha"].min(),
    "→",
    covid["fecha"].max()
)

print(
    "Economía:",
    economia["fecha"].min(),
    "→",
    economia["fecha"].max()
)

print(
    "Turismo:",
    turismo["fecha"].min(),
    "→",
    turismo["fecha"].max()
)

print("\n=== DIMENSIONES ===")

datasets = {
    "Metro": metro,
    "Seguridad": seguridad,
    "COVID": covid,
    "Economía": economia,
    "Turismo": turismo
}

for nombre, df in datasets.items():
    print(f"{nombre:12s}: {df.shape[0]:,} filas x {df.shape[1]} columnas")

=== COBERTURA TEMPORAL ===
Metro: 2018-01-01 00:00:00 → 2024-12-31 00:00:00
Seguridad: 2016-01-01 00:00:00 → 2022-06-30 00:00:00
COVID: 2020-03-24 00:00:00 → 2022-02-09 00:00:00
Economía: 2019-02-28 00:00:00 → 2024-09-30 00:00:00
Turismo: 2018-10-31 00:00:00 → 2024-07-31 00:00:00

=== DIMENSIONES ===
Metro       : 498,615 filas x 6 columnas
Seguridad   : 758,873 filas x 6 columnas
COVID       : 670 filas x 9 columnas
Economía    : 68 filas x 2 columnas
Turismo     : 70 filas x 2 columnas


## 5. Problemas de calidad a documentar

Aquí se identifican problemas que posteriormente justificarán las decisiones de limpieza.


In [6]:
calidad = pd.DataFrame([
    {
        "dataset": "Metro",
        "filas": len(metro),
        "nulos_totales": int(metro.isna().sum().sum()),
        "duplicados_exactos": int(metro.duplicated().sum()),
    },
    {
        "dataset": "Seguridad",
        "filas": len(seguridad),
        "nulos_totales": int(seguridad.isna().sum().sum()),
        "duplicados_exactos": int(seguridad.duplicated().sum()),
    },
    {
        "dataset": "COVID",
        "filas": len(covid),
        "nulos_totales": int(covid.isna().sum().sum()),
        "duplicados_exactos": int(covid.duplicated().sum()),
    },
    {
        "dataset": "Economía",
        "filas": len(economia),
        "nulos_totales": int(economia.isna().sum().sum()),
        "duplicados_exactos": int(economia.duplicated().sum()),
    },
    {
        "dataset": "Turismo",
        "filas": len(turismo),
        "nulos_totales": int(turismo.isna().sum().sum()),
        "duplicados_exactos": int(turismo.duplicated().sum()),
    },
])

calidad


,dataset,filas,nulos_totales,duplicados_exactos
0,Metro,498615,0,0
1,Seguridad,758873,7256,342747
2,COVID,670,0,1
3,Economía,68,0,0
4,Turismo,70,0,0


### 5.1 Revisión de codificación en Metro

En los datos del Metro pueden aparecer variantes como `Linea`, `Línea` o texto con caracteres mal decodificados.  
En esta etapa solo se detectan; la normalización se documentará en la sección de procesamiento.


In [7]:
print("Valores originales de línea:")
print(metro["linea"].value_counts(dropna=False).sort_index())

problemas_codificacion = metro[
    metro["linea"].astype(str).str.contains("Ã|Â", regex=True, na=False)
    | metro["estacion"].astype(str).str.contains("Ã|Â", regex=True, na=False)
]

print(f"Registros con posible problema de codificación: {len(problemas_codificacion):,}")
display(problemas_codificacion.head())


Valores originales de línea:
linea
Linea 1      33520
Linea 12     33520
Linea 2      40224
Linea 3      35196
Linea 4      16760
Linea 5      21788
Linea 6      18436
Linea 7      23464
Linea 8      31844
Linea 9      20112
Linea A      16760
Linea B      35196
LÃ­nea 1     17620
LÃ­nea 12    17620
LÃ­nea 2     21144
LÃ­nea 3     18501
LÃ­nea 4      8810
LÃ­nea 5     11453
LÃ­nea 6      9691
LÃ­nea 7     12334
LÃ­nea 8     16739
LÃ­nea 9     10572
LÃ­nea A      8810
LÃ­nea B     18501
Name: count, dtype: int64
Registros con posible problema de codificación: 276,223


,fecha,anio,mes,linea,estacion,afluencia
0,2018-01-01,2018,Enero,Linea 1,PantitlÃ¡n,16979
2,2018-01-01,2018,Enero,Linea 1,Isabel la CatÃ³lica,7718
5,2018-01-01,2018,Enero,Linea 2,Pino SuÃ¡rez,11418
11,2018-01-01,2018,Enero,Linea 3,Centro MÃ©dico,5184
12,2018-01-01,2018,Enero,Linea 3,EtiopÃ­a/Plaza de la Transparencia,8359


## 6. Filtrado temporal para el proyecto

Para seguridad se utiliza **`anio_inicio` / `fecha_inicio`**, no el año del hecho.  
Esto representa el momento en que la carpeta de investigación fue iniciada.

El análisis principal utiliza 2019–2024. La integración común exacta se ajustará posteriormente según la cobertura mensual de todas las fuentes.


In [8]:
seguridad_2019_2024 = seguridad[
    seguridad["anio_inicio"].between(2019, 2024)
].copy()

metro_2019_2024 = metro[
    metro["fecha"].dt.year.between(2019, 2024)
].copy()

print(f"Seguridad 2019–2024: {len(seguridad_2019_2024):,} registros")
print(f"Metro 2019–2024: {len(metro_2019_2024):,} registros")


Seguridad 2019–2024: 681,024 registros
Metro 2019–2024: 427,440 registros


## 7. Variable de periodo

Se agrega únicamente para facilitar las comparaciones estadísticas.  
No implica que todos los efectos de la pandemia hayan terminado exactamente en 2022.


In [9]:
def clasificar_periodo(anio):
    if anio == 2019:
        return "Pre-COVID"
    elif anio in (2020, 2021):
        return "COVID"
    elif 2022 <= anio <= 2024:
        return "Recuperación"
    return "Fuera del análisis"

# Ejemplo
pd.Series(range(2019, 2025)).to_frame("anio").assign(
    periodo=lambda x: x["anio"].map(clasificar_periodo)
)


,anio,periodo
0,2019,Pre-COVID
1,2020,COVID
2,2021,COVID
3,2022,Recuperación
4,2023,Recuperación
5,2024,Recuperación


## 8. Agregación mensual preliminar

Todavía no se realiza la limpieza definitiva. Esta sección sirve para validar que las fuentes pueden llevarse a una escala temporal común.


In [10]:
# Metro: suma mensual de entradas
metro_mensual = (
    metro_2019_2024
    .assign(mes_fecha=lambda x: x["fecha"].dt.to_period("M").dt.to_timestamp())
    .groupby("mes_fecha", as_index=False)["afluencia"]
    .sum()
    .rename(columns={"afluencia": "afluencia_metro"})
)

# Seguridad: número de carpetas iniciadas por mes
seguridad_mensual = (
    seguridad_2019_2024
    .dropna(subset=["fecha_inicio"])
    .assign(mes_fecha=lambda x: x["fecha_inicio"].dt.to_period("M").dt.to_timestamp())
    .groupby("mes_fecha", as_index=False)
    .size()
    .rename(columns={"size": "carpetas_fgJ"})
)

# Economía y turismo ya son mensuales
economia_mensual = economia.copy()
economia_mensual["mes_fecha"] = economia_mensual["fecha"].dt.to_period("M").dt.to_timestamp()
economia_mensual = economia_mensual[["mes_fecha", "ciudad_de_mexico"]].rename(
    columns={"ciudad_de_mexico": "empleos_formales"}
)

turismo_mensual = turismo.copy()
turismo_mensual["mes_fecha"] = turismo_mensual["fecha"].dt.to_period("M").dt.to_timestamp()
turismo_mensual = turismo_mensual[["mes_fecha", "ciudad_de_mexico"]].rename(
    columns={"ciudad_de_mexico": "ocupacion_hotelera"}
)

display(metro_mensual.head())
display(seguridad_mensual.head())
display(economia_mensual.head())
display(turismo_mensual.head())


,mes_fecha,afluencia_metro
0,2019-01-01,133040079
1,2019-02-01,124568306
2,2019-03-01,132417573
3,2019-04-01,121308154
4,2019-05-01,140358775


,mes_fecha,carpetas_fgJ
0,2019-01-01,21384
1,2019-02-01,20656
2,2019-03-01,22076
3,2019-04-01,20788
4,2019-05-01,22092


,mes_fecha,empleos_formales
0,2019-02-01,14884
1,2019-03-01,12341
2,2019-04-01,6319
3,2019-05-01,21197
4,2019-06-01,2886


,mes_fecha,ocupacion_hotelera
0,2018-10-01,73.17
1,2018-11-01,75.58
2,2018-12-01,65.45
3,2019-01-01,53.13
4,2019-02-01,69.30


## 9. Plan de contingencia: dataset > 1,000 registros

Si el proyecto final exige más de 1,000 observaciones, **no se inventarán datos diarios ni se repetirán artificialmente indicadores mensuales**.

La unidad de observación cambiará a:

> **estación–línea–mes**

Cada estación del Metro aporta una observación mensual. Los indicadores de seguridad, economía y turismo del mismo mes pueden incorporarse como **variables contextuales de ciudad**.

Con el periodo común aproximado de febrero de 2019 a julio de 2024 se esperan alrededor de **12,800 registros**, por lo que este diseño supera ampliamente una eventual exigencia de 1,000 observaciones.

Esta estructura permitiría aplicar clustering o modelos posteriores a nivel estación-mes.


In [11]:
# Comprobación del tamaño potencial del plan de contingencia.
# Esta celda NO construye todavía el dataset final.

def corregir_mojibake(valor):
    if pd.isna(valor):
        return valor
    texto = str(valor)
    if "Ã" in texto or "Â" in texto:
        try:
            return texto.encode("latin1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return texto
    return texto

metro_contingencia = metro.copy()
metro_contingencia["linea_limpia"] = (
    metro_contingencia["linea"]
    .map(corregir_mojibake)
    .str.replace(r"^Linea\b", "Línea", regex=True)
)
metro_contingencia["estacion_limpia"] = metro_contingencia["estacion"].map(corregir_mojibake)

metro_contingencia = metro_contingencia[
    metro_contingencia["fecha"].between("2019-02-01", "2024-07-31")
].copy()

metro_estacion_mes = (
    metro_contingencia
    .assign(mes_fecha=lambda x: x["fecha"].dt.to_period("M").dt.to_timestamp())
    .groupby(
        ["mes_fecha", "linea_limpia", "estacion_limpia"],
        as_index=False
    )["afluencia"]
    .sum()
)

print(f"Registros estación–línea–mes: {len(metro_estacion_mes):,}")
print(f"Meses: {metro_estacion_mes['mes_fecha'].nunique()}")
print(f"Combinaciones estación-línea: {metro_estacion_mes[['linea_limpia','estacion_limpia']].drop_duplicates().shape[0]}")
display(metro_estacion_mes.head())


Registros estación–línea–mes: 12,869
Meses: 66
Combinaciones estación-línea: 197


,mes_fecha,linea_limpia,estacion_limpia,afluencia
0,2019-02-01,Línea 1,Balbuena,367338
1,2019-02-01,Línea 1,Balderas,638686
2,2019-02-01,Línea 1,Boulevard Puerto Aéreo,675875
3,2019-02-01,Línea 1,Candelaria,648318
4,2019-02-01,Línea 1,Chapultepec,1505029


## Siguiente etapa

En la siguiente iteración del notebook se realizará:

1. Normalización definitiva de nombres y codificaciones.
2. Tratamiento justificado de nulos y duplicados.
3. Estadística descriptiva por periodo.
4. Gráficas de evolución temporal.
5. Comparación Pre-COVID / COVID / Recuperación.
6. Integración del dataset procesado.
7. Exportación de una muestra aleatoria de 100 registros.
